In [ ]:
!pip install geopandas shapely rtree fiona pyproj --quiet


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# 🚀 Chemins
GTFS_BASE = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_CLEAN_1"
OSM_FILE = "/content/drive/MyDrive/OSM_Spain_Clean.gpkg"  # ton OSM Espagne
OUTPUT_DIR = "/content/drive/MyDrive/GTFS_FINAL/OSM_MATCHED_1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
!ls "/content/drive/MyDrive"

'Colab Notebooks'
'Consorcio_Regional_de_Transportes_de_Madrid_CRTM_Madrid_City_Bus_(Autobus_urbano_de_Madrid)_DONE.txt'
'Consorcio_Regional_de_Transportes_de_Madrid_CRTM_Madrid_City_Bus_(Autobus_urbano_de_Madrid)_features.parquet'
 en.openfoodfacts.org.products.csv
'exemple rapport.docx'
'Google AI Studio'
 GRAPHS_UPDATED
 GRAPHS_WITH_LINES
 GRAPHS_WITH_TRANSFERS
 GTFS_FINAL
 HALIMA_ELBAHLOULI_CV_PFE.pdf
'Lurraldebus_Guipuzcoana_(La_Guipuzcoana)_DONE.txt'
'Lurraldebus_Guipuzcoana_(La_Guipuzcoana)_features.parquet'
 matchingipynb
 note_laila.jpg
 OSM_Spain_Clean.gpkg


In [ ]:
import fiona
from pathlib import Path
print(fiona.listlayers(OSM_FILE))

['roads_clean']


In [ ]:
# Charger OSM Espagne (layer roads ou celui approprié)
gdf_osm = gpd.read_file(OSM_FILE, layer="roads_clean")


In [ ]:
# Liste des villes à traiter
cities = [d for d in os.listdir(GTFS_BASE) if os.path.isdir(os.path.join(GTFS_BASE, d))]


In [ ]:
for city in cities:
    print(f"\n🔹 Traitement de {city}")

    city_dir = os.path.join(GTFS_BASE, city)
    # Charger tous les fichiers Parquet de la ville
    df_list = [pd.read_parquet(os.path.join(city_dir, f)) for f in os.listdir(city_dir) if f.endswith(".parquet")]
    df_city = pd.concat(df_list, ignore_index=True)
    print(f"  - GTFS edges: {len(df_city)} lignes")

    # Transformer GTFS en GeoDataFrame avec LineString (from → to)
    df_city['geometry'] = df_city.apply(
        lambda row: LineString([(row.from_lon, row.from_lat), (row.to_lon, row.to_lat)]),
        axis=1
    )
    gdf_city = gpd.GeoDataFrame(df_city, geometry='geometry', crs="EPSG:4326")

    # Définir le rectangle englobant du réseau GTFS
    minx, miny, maxx, maxy = gdf_city.total_bounds

    # Sélection des routes OSM dans ce rectangle
    gdf_osm_city = gdf_osm.cx[minx:maxx, miny:maxy].copy()
    print(f"  - OSM extraites: {len(gdf_osm_city)} lignes dans le rectangle de {city}")

    # Sauvegarder le sous-ensemble OSM par ville
    output_path = os.path.join(OUTPUT_DIR, f"{city}_osm_matched.gpkg")
    gdf_osm_city.to_file(output_path, driver="GPKG")
    print(f"  - Sauvegardé dans {output_path}")


🔹 Traitement de Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)
  - GTFS edges: 161 lignes
  - OSM extraites: 1375 lignes dans le rectangle de Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)
  - Sauvegardé dans /content/drive/MyDrive/GTFS_FINAL/OSM_MATCHED_1/Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)_osm_matched.gpkg

🔹 Traitement de Oñati_urbain_(Oñatiko_herribusa)
  - GTFS edges: 140 lignes
  - OSM extraites: 1021 lignes dans le rectangle de Oñati_urbain_(Oñatiko_herribusa)
  - Sauvegardé dans /content/drive/MyDrive/GTFS_FINAL/OSM_MATCHED_1/Oñati_urbain_(Oñatiko_herribusa)_osm_matched.gpkg

🔹 Traitement de Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)
  - GTFS edges: 10872 lignes
  - OSM extraites: 6340 lignes dans le rectangle de Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)
  - Sauvegardé dans /content/drive/MyDrive/GTFS_FINAL/OSM_MATCHED_1/Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_osm_matched.gpkg

🔹 Traitement de TRAM_Alicante
  - GTFS edges: 326

In [ ]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString

# 📂 Dossier de sortie
PARQUET_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_1"
os.makedirs(PARQUET_DIR, exist_ok=True)

# ⚙️ Boucle sur toutes les villes
for city in cities:
    print(f"\n🔹 Traitement de {city}")
    city_dir = os.path.join(GTFS_BASE, city)

    # Charger tous les fichiers Parquet de la ville
    df_list = [
        pd.read_parquet(os.path.join(city_dir, f))
        for f in os.listdir(city_dir)
        if f.endswith(".parquet")
    ]

    if not df_list:
        print(f"⚠ Aucun fichier Parquet trouvé pour {city}, ignoré.")
        continue

    df_city = pd.concat(df_list, ignore_index=True)

    # Recréation de la géométrie LineString (from → to)
    df_city["geometry"] = df_city.apply(
        lambda row: LineString([(row.from_lon, row.from_lat), (row.to_lon, row.to_lat)]),
        axis=1
    )

    # Conversion en GeoDataFrame avec CRS WGS84
    gdf_city = gpd.GeoDataFrame(df_city, geometry="geometry", crs="EPSG:4326")

    # Sauvegarde en Parquet
    output_path = os.path.join(PARQUET_DIR, f"{city}_gtfs_lines.parquet")
    gdf_city.to_parquet(output_path, index=False)

    print(f"✅ GeoDataFrame GTFS de {city} sauvegardé en Parquet : {output_path}")


🔹 Traitement de Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)
✅ GeoDataFrame GTFS de Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani) sauvegardé en Parquet : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_1/Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)_gtfs_lines.parquet

🔹 Traitement de Oñati_urbain_(Oñatiko_herribusa)
✅ GeoDataFrame GTFS de Oñati_urbain_(Oñatiko_herribusa) sauvegardé en Parquet : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_1/Oñati_urbain_(Oñatiko_herribusa)_gtfs_lines.parquet

🔹 Traitement de Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)
✅ GeoDataFrame GTFS de Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres) sauvegardé en Parquet : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_1/Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_gtfs_lines.parquet

🔹 Traitement de TRAM_Alicante
✅ GeoDataFrame GTFS de TRAM_Alicante sauvegardé en Parquet : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_1/TRAM_Alicante_gtfs_lines.parquet

🔹 Tr

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_1"

cities = [
    f.replace("_gtfs_lines.parquet", "")
    for f in os.listdir(BASE_DIR)
    if f.endswith("_gtfs_lines.parquet")
]

print("Villes détectées :", cities)


Villes détectées : ['Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)', 'Oñati_urbain_(Oñatiko_herribusa)', 'Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)', 'TRAM_Alicante', 'Catalonia_Area_de_Barcelona', 'Empresa_Municipal_de_Transportes_de_Madrid_(EMT_Madrid)', 'AUCORSA_(Autobuses_de_Córdoba_S.A.)', 'Sopela_Town_Hall', 'Xunta_de_Galicia_Buses', 'Lurraldebus_Guipuzcoana_(La_Guipuzcoana)', 'Consorcio_Regional_de_Transportes_de_Madrid_CRTM_Madrid_City_Bus_(Autobus_urbano_de_Madrid)', 'Lurraldebus_Tolosaldea', 'Guaguas_Municipales', 'Leioa_City_Council_(Lejoan_Busa)', 'La_Regional_Vallisoletana_SA', 'Transports_Municipaux_D’Egara_(TMESA)_Terrassa_bus_urbain', 'Junta_de_Extremadura_(Bus_du_gouvernement_régional_d’Estrémadure)', 'BlaBlaCar_Bus', 'Direxis_TGO_(Transportes_Generales_de_Olesa)', 'Lurraldebus_Ekialdebus', 'TranspRober', 'TUS_(Transportes_Urbanos_de_Santander)', 'MetroValencia', 'TIB_Transports_of_the_Balearic_Islands_-_CIME_Consell_Insular_de_Menorca_(Menorca_Island_

In [ ]:
# Exemple avec une ville
city = cities[0]
parquet_file = f"/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_1/{city}_gtfs_lines.parquet"

# Charger le GeoDataFrame
gdf_gtfs = gpd.read_parquet(parquet_file)

# Vérifications basiques
print("✅ Aperçu du GeoDataFrame :")
print(gdf_gtfs.head())

print("\n✅ CRS du GeoDataFrame :")
print(gdf_gtfs.crs)

print("\n✅ Exemple de géométrie (LINESTRING) :")
print(gdf_gtfs.geometry.iloc[0])


✅ Aperçu du GeoDataFrame :
         trip_id                                               city  \
0  1_101_1_27600  Lurraldebus - Hernani Urban (bus urbain d’Hern...   
1  1_101_1_27600  Lurraldebus - Hernani Urban (bus urbain d’Hern...   
2  1_101_1_27600  Lurraldebus - Hernani Urban (bus urbain d’Hern...   
3  1_101_1_27600  Lurraldebus - Hernani Urban (bus urbain d’Hern...   
4  1_101_1_27600  Lurraldebus - Hernani Urban (bus urbain d’Hern...   

  route_short_name    route_long_name route_id  route_type from_stop_id  \
0            hern.  urbano de hernani        1           3         4102   
1            hern.  urbano de hernani        1           3         5547   
2            hern.  urbano de hernani        1           3         5549   
3            hern.  urbano de hernani        1           3         1610   
4            hern.  urbano de hernani        1           3         5552   

  to_stop_id   from_lat  from_lon  ...    to_lon  from_arrival_time  \
0       5561  43.265060 

In [ ]:
# Projection GTFS en mètres (UTM 30N – Espagne)
gdf_gtfs_m = gdf_gtfs.to_crs(epsg=25830)

print("✅ CRS projeté :", gdf_gtfs_m.crs)
print("✅ Exemple géométrie projetée :")
print(gdf_gtfs_m.geometry.iloc[0])


✅ CRS projeté : EPSG:25830
✅ Exemple géométrie projetée :
LINESTRING (582872.769934848 4790756.1221552305, 582823.7703472914 4790393.125912922)


In [ ]:


IN_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_1"
OUT_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_UTM_1"

os.makedirs(OUT_DIR, exist_ok=True)

# Déduire la liste des villes depuis les fichiers existants
cities = [
    f.replace("_gtfs_lines.parquet", "")
    for f in os.listdir(IN_DIR)
    if f.endswith("_gtfs_lines.parquet")
]

print(f"📦 Villes détectées : {len(cities)}")

for city in cities:
    print(f"\n🔹 Projection + sauvegarde : {city}")

    in_file = f"{IN_DIR}/{city}_gtfs_lines.parquet"
    out_file = f"{OUT_DIR}/{city}_gtfs_lines_utm30.parquet"

    # Charger
    gdf = gpd.read_parquet(in_file)

    # Vérification minimale
    assert gdf.crs == "EPSG:4326", f"CRS invalide pour {city}"

    # Projection en mètres
    gdf_m = gdf.to_crs(epsg=25830)

    # Sauvegarde
    gdf_m.to_parquet(out_file)

    print(f"✅ Sauvegardé : {out_file}")


📦 Villes détectées : 84

🔹 Projection + sauvegarde : Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)
✅ Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_UTM_1/Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)_gtfs_lines_utm30.parquet

🔹 Projection + sauvegarde : Oñati_urbain_(Oñatiko_herribusa)
✅ Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_UTM_1/Oñati_urbain_(Oñatiko_herribusa)_gtfs_lines_utm30.parquet

🔹 Projection + sauvegarde : Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)
✅ Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_UTM_1/Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_gtfs_lines_utm30.parquet

🔹 Projection + sauvegarde : TRAM_Alicante
✅ Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_UTM_1/TRAM_Alicante_gtfs_lines_utm30.parquet

🔹 Projection + sauvegarde : Catalonia_Area_de_Barcelona
✅ Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_UTM_1/Catalonia_Area_de_Barcelona_gtfs_lines_u

In [ ]:
# Longueur des géométries en mètres
lengths = gdf_m.geometry.length

# Seuil (1 m est raisonnable)
degenerate_edges = gdf_m[lengths <= 1]

print(f"🔎 Nombre d'edges dégénérés (≤ 1 m) : {len(degenerate_edges)}")
print(f"📊 Total edges : {len(gdf_m)}")
print(f"📉 Ratio : {len(degenerate_edges) / len(gdf_m):.2%}")


🔎 Nombre d'edges dégénérés (≤ 1 m) : 0
📊 Total edges : 40
📉 Ratio : 0.00%


In [ ]:
import os
import geopandas as gpd
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_1"

cities = sorted([
    f.replace("_gtfs_lines.parquet", "")
    for f in os.listdir(BASE_DIR)
    if f.endswith("_gtfs_lines.parquet")
])

print(f"🗺️ Nombre de villes détectées : {len(cities)}")


🗺️ Nombre de villes détectées : 84


In [ ]:
results = []

for city in cities:
    print(f"\n🔍 Analyse de {city}")

    file_path = os.path.join(BASE_DIR, f"{city}_gtfs_lines.parquet")
    gdf = gpd.read_parquet(file_path)

    # Projection en mètres
    gdf_m = gdf.to_crs(epsg=25830)

    total_edges = len(gdf_m)

    # Edges dégénérés (≤ 1 m)
    degenerate_count = (gdf_m.geometry.length <= 1).sum()

    ratio = degenerate_count / total_edges if total_edges > 0 else 0

    print(f"  - Total edges : {total_edges}")
    print(f"  - Edges dégénérés : {degenerate_count} ({ratio:.2%})")

    results.append({
        "city": city,
        "total_edges": total_edges,
        "degenerate_edges": degenerate_count,
        "ratio": ratio
    })



🔍 Analyse de AISA_(Bus_Madrid-Aranda_de_Duero-Burgo_de_Osma)
  - Total edges : 326
  - Edges dégénérés : 0 (0.00%)

🔍 Analyse de AUCORSA_(Autobuses_de_Córdoba_S.A.)
  - Total edges : 4858
  - Edges dégénérés : 0 (0.00%)

🔍 Analyse de AUTNA_SL
  - Total edges : 82
  - Edges dégénérés : 0 (0.00%)

🔍 Analyse de Alavabus
  - Total edges : 7635
  - Edges dégénérés : 0 (0.00%)

🔍 Analyse de Alvarez_Travelers_Coaches
  - Total edges : 132
  - Edges dégénérés : 0 (0.00%)

🔍 Analyse de Ancebus
  - Total edges : 46
  - Edges dégénérés : 0 (0.00%)

🔍 Analyse de Auif_Irunbus_(Lurraldebus)
  - Total edges : 141
  - Edges dégénérés : 0 (0.00%)

🔍 Analyse de Autocorb_Coaches
  - Total edges : 1048
  - Edges dégénérés : 0 (0.00%)

🔍 Analyse de Autoridad_de_Transporte_Metropolitano_del_Area_de_Barcelona_(ATM)_Buses_and_trains_in_Catalonia_(full_version)
  - Total edges : 280677
  - Edges dégénérés : 256021 (91.22%)

🔍 Analyse de Avanza_Grupo_(Mataró_city_bus)
  - Total edges : 876
  - Edges dégénéré

In [ ]:
import pandas as pd

# Transformer la liste de dictionnaires en DataFrame
df_results = pd.DataFrame(results)

# Trier par ratio décroissant (villes avec le plus d'edges dégénérés en haut)
df_results_sorted = df_results.sort_values(by="ratio", ascending=False)

# Afficher le tableau
print("\n📊 Résumé qualité des edges par ville :")
print(df_results_sorted)




📊 Résumé qualité des edges par ville :
                                                 city  total_edges  \
28     FGV_-_Generalitat_Valenciana_Metro_de_Valencia        15919   
35                                Guaguas_Municipales         4586   
34            Granada_City_Council_(Granada_city_bus)         5155   
25  Empresa_Municipal_de_Transports_Urbans_de_Palm...          826   
38                                    La_Burundesa_SA         1329   
..                                                ...          ...   
77                                        TranspRober         4979   
79  Transports_Municipaux_D’Egara_(TMESA)_Terrassa...         3300   
80   Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)        10872   
82                             Xunta_de_Galicia_Buses      1522895   
83                                 dBus_(Donostiabus)        14932   

    degenerate_edges     ratio  
28             15919  1.000000  
35              4586  1.000000  
34              5155

OPTIONNEL

In [ ]:
import geopandas as gpd

# Exemple pour une ville
city = "MetroValencia"
parquet_file = f"/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_UTM/{city}_gtfs_lines_utm30.parquet"

# Charger le GeoDataFrame projeté
gdf_city = gpd.read_parquet(parquet_file)

# Calculer la longueur des edges
gdf_city['length_m'] = gdf_city.geometry.length
degenerate_edges = gdf_city[gdf_city.length_m == 0]

print(f"Total edges : {len(gdf_city)}")
print(f"Edges dégénérés : {len(degenerate_edges)} ({len(degenerate_edges)/len(gdf_city):.2%})")


In [ ]:
import os
import pandas as pd

# Chemin vers GTFS
GTFS_BASE = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES"

city = "La_Burundesa_SA"
city_dir = os.path.join(GTFS_BASE, city)

# Charger tous les fichiers Parquet de la ville
df_list = [pd.read_parquet(os.path.join(city_dir, f))
           for f in os.listdir(city_dir) if f.endswith(".parquet")]
df_city = pd.concat(df_list, ignore_index=True)

# Extraire uniquement les coordonnées
coords_df = df_city[['from_lat','from_lon','to_lat','to_lon']]

print(f"Total edges : {len(coords_df)}")
print(coords_df.head(10))


In [ ]:
FIN ***


In [ ]:
# Dossier contenant les fichiers GeoDataFrame projetés en UTM
GTFS_BASE = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_GEO_UTM_1"
OUTPUT_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_BUFFERS_1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Villes à ignorer
IGNORE_CITIES = ["La_Burundesa_SA", "MetroValencia", "TranspRober"]

# Liste des villes présentes
cities = [f.replace("_gtfs_lines_utm30.parquet","")
          for f in os.listdir(GTFS_BASE) if f.endswith(".parquet")]



In [ ]:
for city in cities:
    if city in IGNORE_CITIES:
        print(f"⚠ Ignoré {city} (ville problématique)")
        continue

    print(f"\n🔹 Traitement de {city}")

    # Charger le GeoDataFrame projeté
    file_path = os.path.join(GTFS_BASE, f"{city}_gtfs_lines_utm30.parquet")
    gdf_city = gpd.read_parquet(file_path)
    print(f"  - Chargé {len(gdf_city)} edges")

    # Vérification basique
    print("  - Exemple de géométrie:", gdf_city.geometry.iloc[0])

    # Créer le buffer (30 mètres)
    gdf_city["geometry_buffer"] = gdf_city.geometry.buffer(30)
    print(f"  - Buffers créés (30 m)")

    # Sauvegarder le GeoDataFrame avec buffer en parquet
    output_file = os.path.join(OUTPUT_DIR, f"{city}_gtfs_lines_buffer.parquet")
    gdf_city.to_parquet(output_file)
    print(f"  - Sauvegardé : {output_file}")

print("\n✅ Tous les buffers traités et sauvegardés")


🔹 Traitement de Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)
  - Chargé 161 edges
  - Exemple de géométrie: LINESTRING (582872.769934848 4790756.1221552305, 582823.7703472914 4790393.125912922)
  - Buffers créés (30 m)
  - Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_BUFFERS_1/Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)_gtfs_lines_buffer.parquet

🔹 Traitement de Oñati_urbain_(Oñatiko_herribusa)
  - Chargé 140 edges
  - Exemple de géométrie: LINESTRING (548091.1237838396 4765007.395766746, 547726.127519261 4764690.399116162)
  - Buffers créés (30 m)
  - Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_BUFFERS_1/Oñati_urbain_(Oñatiko_herribusa)_gtfs_lines_buffer.parquet

🔹 Traitement de Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)
  - Chargé 10872 edges
  - Exemple de géométrie: LINESTRING (209818.73619415046 4374990.087817267, 209705.0279914746 4375389.703391444)
  - Buffers créés (30 m)
  - Sauvegardé : /content/drive/MyDrive/GTFS_FINAL

In [ ]:
import geopandas as gpd
import pandas as pd
import os
import gc

GTFS_BUFFER_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_BUFFERS_1"
OSM_DIR = "/content/drive/MyDrive/GTFS_FINAL/OSM_MATCHED_1"
MATCHED_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_MATCHED_1"
os.makedirs(MATCHED_DIR, exist_ok=True)

skip_cities = ["La_Burundesa_SA", "MetroValencia", "TranspRober","Consorcio_Regional_de_Transportes_de_Madrid_CRTM_Intercity_Buses_(Madrid_Intercity_Bus)"]
CHUNK_SIZE = 20000

# ---------------------------
# 1. Villes GTFS disponibles
# ---------------------------
gtfs_files = [
    f for f in os.listdir(GTFS_BUFFER_DIR)
    if f.endswith(".parquet")
]

cities = [
    f.replace("_gtfs_lines_buffer.parquet", "")
    for f in gtfs_files
    if f.replace("_gtfs_lines_buffer.parquet", "") not in skip_cities
]

# ---------------------------
# 2. Villes déjà traitées
# ---------------------------
processed_cities = {
    f.replace("_gtfs_osm_matched.parquet", "")
    for f in os.listdir(MATCHED_DIR)
    if f.endswith("_gtfs_osm_matched.parquet")
}

cities_to_process = [c for c in cities if c not in processed_cities]

print(f"✔ Villes totales : {len(cities)}")
print(f"✔ Déjà traitées : {len(processed_cities)}")
print(f"▶ À traiter     : {len(cities_to_process)}\n")

# ---------------------------
# 3. Traitement ville par ville
# ---------------------------
for city_name in cities_to_process:
    print(f"🔹 Traitement de {city_name}")

    try:
        # --- Charger GTFS buffer
        gtfs_path = os.path.join(
            GTFS_BUFFER_DIR,
            f"{city_name}_gtfs_lines_buffer.parquet"
        )
        gtfs_gdf = gpd.read_parquet(gtfs_path)

        # --- Charger OSM
        osm_path = os.path.join(OSM_DIR, f"{city_name}_osm_matched.gpkg")
        if not os.path.exists(osm_path):
            print(f"❌ OSM manquant pour {city_name}, ignorée\n")
            del gtfs_gdf
            gc.collect()
            continue

        osm_gdf = gpd.read_file(osm_path)

        # --- CRS
        if gtfs_gdf.crs != osm_gdf.crs:
            osm_gdf = osm_gdf.to_crs(gtfs_gdf.crs)

        # --- Réduction bbox
        minx, miny, maxx, maxy = gtfs_gdf.total_bounds
        osm_gdf = osm_gdf.cx[minx:maxx, miny:maxy]

        # --- Spatial join par chunks
        matched_chunks = []

        for start in range(0, len(osm_gdf), CHUNK_SIZE):
            osm_chunk = osm_gdf.iloc[start:start + CHUNK_SIZE]
            matched_chunk = gpd.sjoin(
                osm_chunk,
                gtfs_gdf,
                how="inner",
                predicate="intersects"
            )

            if not matched_chunk.empty:
                matched_chunks.append(matched_chunk)

            print(f"    Chunk {start}-{start+len(osm_chunk)} | matchés: {len(matched_chunk)}")

            del osm_chunk, matched_chunk
            gc.collect()

        if not matched_chunks:
            print(f"⚠ Aucun matching pour {city_name}\n")
            del gtfs_gdf, osm_gdf
            gc.collect()
            continue

        # --- Concat final
        matched_gdf = gpd.GeoDataFrame(
            pd.concat(matched_chunks, ignore_index=True),
            crs=gtfs_gdf.crs
        )

        # --- Sauvegarde finale (UNE FOIS)
        out_path = os.path.join(
            MATCHED_DIR,
            f"{city_name}_gtfs_osm_matched.parquet"
        )
        matched_gdf.to_parquet(out_path)
        print(f"✅ Sauvegardé : {out_path}\n")

        del gtfs_gdf, osm_gdf, matched_gdf, matched_chunks
        gc.collect()

    except Exception as e:
        print(f"💥 Crash sur {city_name} : {e}")
        print("➡ La ville sera retraitée au prochain run\n")
        gc.collect()
        continue


✔ Villes totales : 80
✔ Déjà traitées : 0
▶ À traiter     : 80

🔹 Traitement de Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)
    Chunk 0-1366 | matchés: 1719
✅ Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_MATCHED_1/Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)_gtfs_osm_matched.parquet

🔹 Traitement de Oñati_urbain_(Oñatiko_herribusa)
    Chunk 0-1013 | matchés: 1350
✅ Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_MATCHED_1/Oñati_urbain_(Oñatiko_herribusa)_gtfs_osm_matched.parquet

🔹 Traitement de Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)
    Chunk 0-6324 | matchés: 89320
✅ Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_MATCHED_1/Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_gtfs_osm_matched.parquet

🔹 Traitement de TRAM_Alicante
    Chunk 0-20000 | matchés: 9280
    Chunk 20000-40000 | matchés: 9734
    Chunk 40000-44908 | matchés: 1376
✅ Sauvegardé : /content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_MATCHED_1/TRAM_Ali

In [ ]:
import os

MATCHED_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_MATCHED_1"

files = [
    f for f in os.listdir(MATCHED_DIR)
    if f.endswith("_gtfs_osm_matched.parquet")
]

print(f"📦 Fichiers disponibles : {len(files)}")
files[:5]


📦 Fichiers disponibles : 71


['Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)_gtfs_osm_matched.parquet',
 'Oñati_urbain_(Oñatiko_herribusa)_gtfs_osm_matched.parquet',
 'Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_gtfs_osm_matched.parquet',
 'TRAM_Alicante_gtfs_osm_matched.parquet',
 'Catalonia_Area_de_Barcelona_gtfs_osm_matched.parquet']

In [ ]:
import geopandas as gpd

city = files[0]  # ou choisis-en une à la main
path = os.path.join(MATCHED_DIR, city)

gdf = gpd.read_parquet(path)

print("✅ Aperçu :")
gdf.head(2)


✅ Aperçu :


,id,highway,junction,lanes,maxspeed,oneway,busway,psv,motorcar,access,...,to_lon,from_arrival_time,from_departure_time,to_arrival_time,to_departure_time,travel_time_sec,travel_time_departure_sec,delta_lat,delta_lon,geometry_buffer
0,198612598,secondary,None,2,None,no,None,None,None,None,...,-1.979522,07:54:24,07:54:24,07:55:17,07:55:17,53,53,-0.003263,-0.000658,"POLYGON ((582853.501 4790389.113, 582852.964 4..."
1,24059150,secondary,roundabout,1,None,no,None,None,None,None,...,-1.980811,07:51:07,07:51:07,07:52:24,07:52:24,77,77,-0.002935,-0.004103,"POLYGON ((582730.017 4791218.936, 582727.839 4..."


In [ ]:
print("CRS :", gdf.crs)
print("Colonnes :", list(gdf.columns))
print("Type :", type(gdf))
print("Exemple géométrie :", gdf.geometry.iloc[0])


CRS : {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "ETRS89 / UTM zone 30N", "base_crs": {"name": "ETRS89", "datum_ensemble": {"name": "European Terrestrial Reference System 1989 ensemble", "members": [{"name": "European Terrestrial Reference Frame 1989"}, {"name": "European Terrestrial Reference Frame 1990"}, {"name": "European Terrestrial Reference Frame 1991"}, {"name": "European Terrestrial Reference Frame 1992"}, {"name": "European Terrestrial Reference Frame 1993"}, {"name": "European Terrestrial Reference Frame 1994"}, {"name": "European Terrestrial Reference Frame 1996"}, {"name": "European Terrestrial Reference Frame 1997"}, {"name": "European Terrestrial Reference Frame 2000"}, {"name": "European Terrestrial Reference Frame 2005"}, {"name": "European Terrestrial Reference Frame 2014"}, {"name": "European Terrestrial Reference Frame 2020"}], "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 2

In [ ]:
gdf["length_m"] = gdf.geometry.length

print("Longueur min :", gdf.length_m.min())
print("Longueur médiane :", gdf.length_m.median())
print("Longueur max :", gdf.length_m.max())


In [ ]:
import geopandas as gpd

osm_file = "/content/drive/MyDrive/GTFS_FINAL/OSM_MATCHED_1/AISA_(Bus_Madrid-Aranda_de_Duero-Burgo_de_Osma)_osm_matched.gpkg"
osm_gdf = gpd.read_file(osm_file)

print(osm_gdf.crs)


EPSG:4326


In [ ]:
epsg=25830

In [ ]:


import geopandas as gpd

osm_file = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_BUFFERS/AISA_(Bus_Madrid-Aranda_de_Duero-Burgo_de_Osma)_gtfs_lines_buffer.parquet"
osm_gdf = gpd.read_parquet(osm_file)

print(osm_gdf.crs)


{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "ETRS89 / UTM zone 30N", "base_crs": {"name": "ETRS89", "datum_ensemble": {"name": "European Terrestrial Reference System 1989 ensemble", "members": [{"name": "European Terrestrial Reference Frame 1989"}, {"name": "European Terrestrial Reference Frame 1990"}, {"name": "European Terrestrial Reference Frame 1991"}, {"name": "European Terrestrial Reference Frame 1992"}, {"name": "European Terrestrial Reference Frame 1993"}, {"name": "European Terrestrial Reference Frame 1994"}, {"name": "European Terrestrial Reference Frame 1996"}, {"name": "European Terrestrial Reference Frame 1997"}, {"name": "European Terrestrial Reference Frame 2000"}, {"name": "European Terrestrial Reference Frame 2005"}, {"name": "European Terrestrial Reference Frame 2014"}, {"name": "European Terrestrial Reference Frame 2020"}], "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257

In [ ]:
##hna kan ri dak score matching distance_m,overlap.. ma7tajinahum f waaaaaalu

In [ ]:
import geopandas as gpd
import os

MATCHING_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_MATCHED_1"

# Lister les fichiers disponibles
files = [f for f in os.listdir(MATCHING_DIR) if f.endswith("_gtfs_osm_matched.parquet")]
print("📂 Fichiers disponibles :", len(files))
print(files[:5])   # affiche les 5 premiers noms de fichiers

# Choisir une ville (par exemple le premier fichier)
city_file = files[0]
path = os.path.join(MATCHING_DIR, city_file)

# Charger et afficher les 5 premières lignes
gdf = gpd.read_parquet(path)
print(f"\n✅ Aperçu de {city_file} :")
print(gdf.head())


📂 Fichiers disponibles : 71
['Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)_gtfs_osm_matched.parquet', 'Oñati_urbain_(Oñatiko_herribusa)_gtfs_osm_matched.parquet', 'Vectalia_Movilidad_(bus_de_la_ville_de_Cáceres)_gtfs_osm_matched.parquet', 'TRAM_Alicante_gtfs_osm_matched.parquet', 'Catalonia_Area_de_Barcelona_gtfs_osm_matched.parquet']

✅ Aperçu de Lurraldebus_-_Hernani_Urban_(bus_urbain_d’Hernani)_gtfs_osm_matched.parquet :
          id    highway    junction  lanes maxspeed oneway busway   psv  \
0  198612598  secondary        None      2     None     no   None  None   
1   24059150  secondary  roundabout      1     None     no   None  None   
2   24059150  secondary  roundabout      1     None     no   None  None   
3   24059150  secondary  roundabout      1     None     no   None  None   
4   24059150  secondary  roundabout      1     None     no   None  None   

  motorcar access  ...    to_lon from_arrival_time from_departure_time  \
0     None   None  ... -1.979522      

In [ ]:
import os
import pandas as pd

MATCHED_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_MATCHED_1"

stats = []

for f in os.listdir(MATCHED_DIR):
    if not f.endswith("_gtfs_osm_matched.parquet"):
        continue

    city = f.replace("_gtfs_osm_matched.parquet", "")
    path = os.path.join(MATCHED_DIR, f)

    try:
        # Compter les lignes directement
        n_rows = pd.read_parquet(path).shape[0]
        file_size_mb = os.path.getsize(path) / (1024 * 1024)

        stats.append({
            "city": city,
            "rows": n_rows,
            "file_mb": round(file_size_mb, 2)
        })

    except Exception as e:
        print(f"❌ Erreur sur {city} : {e}")

# Créer le DataFrame seulement si stats n’est pas vide
if stats:
    df_stats = pd.DataFrame(stats).sort_values("rows", ascending=False)
    print("✅ Nombre total de lignes par ville :")
    print(df_stats.head(10))   # affiche les 10 premières villes
else:
    print("⚠ Aucun fichier n’a pu être lu, stats est vide.")


✅ Nombre total de lignes par ville :
                                                 city     rows  file_mb
8                              Xunta_de_Galicia_Buses  8953920  1725.10
25                         La_Coruña_Tram_Company_SA  2201944    40.28
4                         Catalonia_Area_de_Barcelona  1827623   336.81
5   Empresa_Municipal_de_Transportes_de_Madrid_(EM...   933635   262.83
62                                          Bizkaibus   701780   109.13
10  Consorcio_Regional_de_Transportes_de_Madrid_CR...   699936    71.28
16                                      BlaBlaCar_Bus   512348    29.92
28           Generalitat_of_Catalonia_(Intercity_bus)   346997   133.28
51                                 dBus_(Donostiabus)   281188     4.97
32  Autoridad_de_Transporte_Metropolitano_del_Area...   199559    71.62


In [ ]:
import geopandas as gpd
import pandas as pd
import os
import gc
import numpy as np

# ===============================
# CONFIG
# ===============================
FEATURE_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_MATCHED_1"
COST_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_COSTS_1"
os.makedirs(COST_DIR, exist_ok=True)
CHUNK_SIZE = 20000

# ===============================
# FONCTION MULTI-COST AMÉLIORÉE
# ===============================
def compute_multi_cost(df):
    df = df.copy()
    # --- 1️⃣ VITESSE
    df["maxspeed_kmh"] = (
        df["maxspeed"]
        .astype(str)
        .str.extract(r"(\d+)")
        .astype(float)
        .fillna(30)
        .clip(lower=10, upper=90)
    )
    speed_mps = df["maxspeed_kmh"] * 1000 / 3600

    # --- 2️⃣ DISTANCE
    df["distance_m_real"] = df.geometry.length

    # --- 3️⃣ TEMPS
    df["cost_time_s"] = df["distance_m_real"] / speed_mps

    # --- 4️⃣ JUNCTION
    df["cost_intersection"] = (
        df["junction"]
        .map({"roundabout": 8, "circular": 6, "yes": 5})
        .fillna(0)
    )

    # --- 5️⃣ HIGHWAY
    df["cost_highway"] = (
        df["highway"]
        .map({"residential": 9, "tertiary": 7, "secondary": 7,
              "primary": 1, "trunk": 1, "busway": 1})
        .fillna(3)
    )

    # --- 6️⃣ LANES
    df["cost_lanes"] = df["lanes"].apply(lambda x: 4 if pd.notna(x) and x >= 3 else 1)

    # --- 7️⃣ ONEWAY
    def oneway_cost(val):
        if pd.isna(val): return 3
        val = str(val).lower()
        if val in ["yes", "-1"]: return 1
        elif val == "no": return 3
        elif val in ["alternating", "reversible"]: return 6
        else: return 3
    df["cost_oneway"] = df["oneway"].apply(oneway_cost)

    # --- 8️⃣ MOTORCAR
    df["cost_motorcar"] = df["motorcar"].apply(lambda x: 1 if pd.notna(x) and str(x).lower() == "no" else 4)

    # --- 9️⃣ COÛT TOTAL
    df["cost_total"] = (
        0.4 * (df["cost_time_s"] / 60) +
        0.15 * (df["distance_m_real"] / 1000) +
        0.15 * df["cost_intersection"] +
        0.1 * df["cost_highway"] +
        0.1 * df["cost_lanes"] +
        0.05 * df["cost_oneway"] +
        0.05 * df["cost_motorcar"]
    )
    return df

# ===============================
# 1️⃣ LISTE DES VILLES
# ===============================
cities = [f.replace(".parquet", "") for f in os.listdir(FEATURE_DIR) if f.endswith(".parquet")]
done_cities = {f.replace("_DONE.txt", "") for f in os.listdir(COST_DIR) if f.endswith("_DONE.txt")}
IGNORE_CITIES = ["Xunta_de_Galicia_Buses_gtfs_osm_matched","Catalonia_Area_de_Barcelona_gtfs_osm_matched"]
cities_to_process = [c for c in cities if c not in done_cities and c not in IGNORE_CITIES]

print(f"✔ Total villes       : {len(cities)}")
print(f"✔ Villes terminées   : {len(done_cities)}")
print(f"▶ Villes à traiter   : {len(cities_to_process)}")

# ===============================
# 2️⃣ TRAITEMENT VILLE PAR VILLE AVEC REPRISE
# ===============================
for city in cities_to_process:
    print(f"\n🔹 Calcul des coûts pour {city}")

    input_path = os.path.join(FEATURE_DIR, f"{city}.parquet")
    done_flag = os.path.join(COST_DIR, f"{city}_DONE.txt")

    try:
        gdf = gpd.read_parquet(input_path)
        gdf = gdf.to_crs(epsg=3857)  # Reprojection

        total_rows = len(gdf)
        chunk_idx = 0

        # --- Vérifier les chunks déjà existants pour reprise
        existing_chunks = [
            f for f in os.listdir(COST_DIR)
            if f.startswith(f"{city}_cost_chunk_") and f.endswith(".parquet")
        ]
        if existing_chunks:
            existing_indices = [int(f.split("_")[-1].split(".")[0]) for f in existing_chunks]
            chunk_idx = max(existing_indices) + 1
            start = chunk_idx * CHUNK_SIZE
        else:
            start = 0

        # --- Traitement des chunks restants
        for start in range(start, total_rows, CHUNK_SIZE):
            chunk = gdf.iloc[start:start + CHUNK_SIZE].copy()
            chunk = compute_multi_cost(chunk)

            out_file = os.path.join(COST_DIR, f"{city}_cost_chunk_{chunk_idx}.parquet")
            chunk.to_parquet(out_file)
            print(f"  ✔ Chunk {chunk_idx} sauvegardé ({start} → {start+len(chunk)})")

            chunk_idx += 1
            del chunk
            gc.collect()

        # --- Flag ville terminée
        with open(done_flag, "w") as f:
            f.write("ok")
        print(f"✅ Ville {city} complètement traitée")

        del gdf
        gc.collect()

    except Exception as e:
        print(f"💥 Crash sur {city} : {e}")
        gc.collect()


✔ Total villes       : 71
✔ Villes terminées   : 23
▶ Villes à traiter   : 46

🔹 Calcul des coûts pour La_Coruña_Tram_Company_SA_gtfs_osm_matched


In [ ]:
##fin

In [ ]:
import pandas as pd
import os

# -----------------------------
# 1️⃣ Sélectionner un chunk à vérifier
# -----------------------------
COST_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_COSTS_1"
example_file = [f for f in os.listdir(COST_DIR) if f.endswith(".parquet")][1]  # premier chunk trouvé
file_path = os.path.join(COST_DIR, example_file)

# -----------------------------
# 2️⃣ Lire le chunk
# -----------------------------
df_chunk = pd.read_parquet(file_path)

# -----------------------------
# 3️⃣ Vérification rapide des coûts
# -----------------------------
print("Exemple de données :")
print(df_chunk.head())

# -----------------------------
# 4️⃣ Statistiques des coûts
# -----------------------------
# Adapter les noms des colonnes exactes
cost_columns = [
    "cost_time_s",       # temps en secondes
    "distance_m_real",   # distance en mètres (réelle)

    "cost_intersection", # pénalité intersections
    "cost_total"         # coût total
]

print("\nStatistiques des coûts :")
print(df_chunk[cost_columns].describe())

# -----------------------------
# 5️⃣ Vérification d'unités et cohérence
# -----------------------------
print("\n✅ Vérification unités et valeurs :")
print(f"Temps (s) min/max : {df_chunk['cost_time_s'].min():.2f} / {df_chunk['cost_time_s'].max():.2f}")
print(f"Distance (m) min/max : {df_chunk['distance_m_real'].min():.2f} / {df_chunk['distance_m_real'].max():.2f}")
print(f"Coût total min/max : {df_chunk['cost_total'].min():.2f} / {df_chunk['cost_total'].max():.2f}")
#print(f"Pénalités geom min/max : {df_chunk['cost_geom'].min():.2f} / {df_chunk['cost_geom'].max():.2f}")
print(f"Pénalités intersection min/max : {df_chunk['cost_intersection'].min():.2f} / {df_chunk['cost_intersection'].max():.2f}")


Exemple de données :
         id      highway junction  lanes maxspeed oneway busway   psv  \
0  13433010  residential     None      1     None    yes   None  None   
1  13433010  residential     None      1     None    yes   None  None   
2  13435463  residential     None      1     None     no   None  None   
3  13435463  residential     None      1     None     no   None  None   
4  13435463  residential     None      1     None     no   None  None   

  motorcar access  ... travel_time_sec travel_time_departure_sec delta_lat  \
0     None   None  ...             133                       133  0.003924   
1     None   None  ...             133                       133  0.003924   
2     None   None  ...              55                        55 -0.001258   
3     None   None  ...              55                        55 -0.001258   
4     None   None  ...             142                       142  0.003167   

   delta_lon                                    geometry_buffer  maxspe

In [ ]:
import pandas as pd
import os

# ===============================
# CONFIG
# ===============================
COST_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_COSTS_1"

# ===============================
# 1️⃣ Lister tous les fichiers parquet
# ===============================
cost_files = [
    os.path.join(COST_DIR, f)
    for f in os.listdir(COST_DIR)
    if f.endswith(".parquet")
]

print(f"📦 Fichiers de coûts trouvés : {len(cost_files)}")

# ===============================
# 2️⃣ Lire uniquement la colonne junction (PANDAS)
# ===============================
junction_values = []

for file in cost_files:
    try:
        df = pd.read_parquet(file, columns=["route_type"])
        junction_values.append(df["route_type"])
    except Exception as e:
        print(f"⚠️ Erreur lecture {os.path.basename(file)} : {e}")

# ===============================
# 3️⃣ Concaténer toutes les valeurs
# ===============================
junction_series = pd.concat(junction_values, ignore_index=True)

# ===============================
# 4️⃣ Valeurs DISTINCTES
# ===============================
distinct_junctions = junction_series.dropna().unique()

print("\n🚦 Valeurs DISTINCTES de route_type :\n")
for val in distinct_junctions:
    print(f"- {val}")

print(f"\n✔ Total valeurs distinctes (hors NaN) : {len(distinct_junctions)}")


📦 Fichiers de coûts trouvés : 539

🚦 Valeurs DISTINCTES de route_type :

- 3.0
- 1.0
- 0.0
- 2.0
- 4.0
- 7.0

✔ Total valeurs distinctes (hors NaN) : 6


In [ ]:
import pandas as pd
import os

# ===============================
# CONFIG
# ===============================
COST_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_COSTS_1"

COLUMNS_TO_ANALYZE = [
    "highway",
    "lanes",
    "oneway",
    "psv",
    "motorcar",
    "access"
]

# ===============================
# 1️⃣ Lister tous les fichiers parquet
# ===============================
cost_files = [
    os.path.join(COST_DIR, f)
    for f in os.listdir(COST_DIR)
    if f.endswith(".parquet")
]

print(f"📦 Fichiers de coûts trouvés : {len(cost_files)}")

# ===============================
# 2️⃣ Dictionnaire pour stocker les valeurs
# ===============================
values_dict = {col: [] for col in COLUMNS_TO_ANALYZE}

# ===============================
# 3️⃣ Lecture fichiers
# ===============================
for file in cost_files:
    try:
        df = pd.read_parquet(file, columns=COLUMNS_TO_ANALYZE)

        for col in COLUMNS_TO_ANALYZE:
            if col in df.columns:
                values_dict[col].append(df[col])

    except Exception as e:
        print(f"⚠️ Erreur lecture {os.path.basename(file)} : {e}")

# ===============================
# 4️⃣ Affichage des valeurs DISTINCTES
# ===============================
print("\n🚦 VALEURS DISTINCTES PAR COLONNE\n")

for col, series_list in values_dict.items():
    if not series_list:
        print(f"❌ {col} : aucune donnée")
        continue

    merged = pd.concat(series_list, ignore_index=True)
    distinct_vals = merged.dropna().unique()

    print(f"🔹 {col} :")
    for val in distinct_vals:
        print(f"   - {val}")
    print(f"   ✔ Total distinct (hors NaN) : {len(distinct_vals)}\n")


📦 Fichiers de coûts trouvés : 539

🚦 VALEURS DISTINCTES PAR COLONNE

🔹 highway :
   - secondary
   - residential
   - tertiary
   - motorway
   - primary
   - trunk
   - busway
   ✔ Total distinct (hors NaN) : 7

🔹 lanes :
   - 2
   - 1
   - 3
   - 4
   - 5
   - 7
   - 6
   - 8
   - 11
   - 9
   ✔ Total distinct (hors NaN) : 10

🔹 oneway :
   - no
   - yes
   - alternating
   - -1
   - reversible
   ✔ Total distinct (hors NaN) : 5

🔹 psv :
   - yes
   - designated
   - no
   - taxi
   ✔ Total distinct (hors NaN) : 4

🔹 motorcar :
   - no
   - yes
   - designated
   - delivery
   - permit
   - private
   ✔ Total distinct (hors NaN) : 6

🔹 access :
   - private
   - destination
   - yes
   - designated
   - permissive
   - no
   - psv
   - police
   - permit
   - emergency
   - restricted
   ✔ Total distinct (hors NaN) : 11



In [ ]:
import pandas as pd
import os

# ===============================
# CONFIG
# ===============================
COST_DIR = "/content/drive/MyDrive/GTFS_FINAL/NETWORK_EDGES_COSTS_1"

# ===============================
# LISTER TOUS LES FICHIERS PARQUET
# ===============================
cost_files = [os.path.join(COST_DIR, f) for f in os.listdir(COST_DIR) if f.endswith(".parquet")]

print(f"📦 Nombre de fichiers Parquet : {len(cost_files)}\n")

# ===============================
# COLLECTE DES COLONNES
# ===============================
all_columns = set()

for file in cost_files:
    try:
        df = pd.read_parquet(file, engine='pyarrow')  # Lire sans géométrie
        all_columns.update(df.columns)
    except Exception as e:
        print(f"⚠️ Erreur lecture {os.path.basename(file)} : {e}")

# ===============================
# AFFICHAGE DES COLONNES
# ===============================
print("📋 Colonnes présentes dans tous les fichiers :\n")
for col in sorted(all_columns):
    print(f"- {col}")

print(f"\n✔ Total colonnes distinctes : {len(all_columns)}")


📦 Nombre de fichiers Parquet : 539

📋 Colonnes présentes dans tous les fichiers :

- access
- busway
- city
- cost_highway
- cost_intersection
- cost_lanes
- cost_motorcar
- cost_oneway
- cost_time_s
- cost_total
- delta_lat
- delta_lon
- distance_m_real
- from_arrival_time
- from_departure_time
- from_lat
- from_lon
- from_stop_id
- geometry
- geometry_buffer
- highway
- id
- index_right
- junction
- lanes
- length
- maxspeed
- maxspeed_kmh
- motorcar
- name
- oneway
- psv
- route_id
- route_long_name
- route_short_name
- route_type
- smoothness
- to_arrival_time
- to_departure_time
- to_lat
- to_lon
- to_stop_id
- travel_time_departure_sec
- travel_time_sec
- trip_id
- width

✔ Total colonnes distinctes : 46
